# Day 2: Concepts, Labels, Dictionaries, and Supervised Classification

Today connects concepts to labels. We start with transparent rule-based measurement and then compare it to supervised classifiers.

By the end you should be able to:

1. Build a small dictionary or regex baseline.
2. Evaluate a classifier with precision, recall, F1, and a confusion matrix.
3. Compare dictionary, Naive Bayes, and logistic regression.
4. Inspect errors as part of measurement validation.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

pd.set_option('display.max_colwidth', 140)

In [ ]:
from pathlib import Path


def find_data_dir():
    candidates = [Path('../data'), Path('materials/data'), Path('data')]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find the course data directory.')


DATA_DIR = find_data_dir()
DATA_DIR

## spaCy setup

The notebooks use spaCy for tokenization and preprocessing. If `en_core_web_sm` is installed, the same setup also shows POS tags, lemmas, and dependency parses. If not, the notebooks still run with a blank English tokenizer.

In [ ]:
import spacy

try:
    nlp = spacy.load('en_core_web_sm')
    print('Loaded spaCy model: en_core_web_sm')
except OSError:
    nlp = spacy.blank('en')
    nlp.add_pipe('sentencizer')
    print('Using spacy.blank("en"). Install en_core_web_sm for POS tags, lemmas, and dependency parses.')


def spacy_preprocess(text, remove_stop=True, keep_alpha=True, min_len=2):
    """Tokenize and lightly preprocess text with spaCy."""
    doc = nlp.make_doc(str(text))
    tokens = []
    for token in doc:
        if keep_alpha and not token.is_alpha:
            continue
        if remove_stop and token.is_stop:
            continue
        term = token.text.lower()
        if len(term) >= min_len:
            tokens.append(term)
    return tokens


def spacy_analyzer(text):
    return spacy_preprocess(text)


def spacy_analyzer_with_bigrams(text):
    tokens = spacy_preprocess(text)
    bigrams = [tokens[i] + '_' + tokens[i + 1] for i in range(len(tokens) - 1)]
    return tokens + bigrams


def spacy_token_table(text):
    """Show tokenization, preprocessing flags, and parses when a full model is available."""
    doc = nlp(str(text))
    rows = []
    for token in doc:
        rows.append({
            'text': token.text,
            'lemma': token.lemma_ or '(model needed)',
            'pos': token.pos_ or '(model needed)',
            'dep': token.dep_ or '(model needed)',
            'is_alpha': token.is_alpha,
            'is_stop': token.is_stop,
            'kept_by_preprocess': token.text.lower() in spacy_preprocess(token.text, remove_stop=True)
        })
    return pd.DataFrame(rows)

## 1. Dictionary measurement on headlines

A dictionary is useful when the concept can be stated as explicit search rules. It is also easy to audit.

In [ ]:
headlines = pd.read_csv(DATA_DIR / 'headlines.csv')
headlines[['source', 'source_type', 'headline']].head()

## 1a. Inspect one headline with spaCy

For dictionary work, spaCy is useful for seeing tokens and basic preprocessing decisions before writing search rules.

In [ ]:
spacy_token_table(headlines.loc[0, 'headline'])

In [ ]:
immigration_terms = [
    'asylum', 'immigrant', 'immigration', 'migrant', 'migrants',
    'refugee', 'refugees', 'border', 'deport', 'deported'
]

pattern = re.compile(r'\b(' + '|'.join(map(re.escape, immigration_terms)) + r')\b', re.IGNORECASE)

headlines['dictionary_hit'] = headlines['headline'].fillna('').str.contains(pattern)
headlines['matched_terms'] = headlines['headline'].fillna('').apply(lambda x: sorted(set(t.lower() for t in pattern.findall(x))))

headlines[['headline', 'dictionary_hit', 'matched_terms']].head(12)

In [ ]:
print(headlines['dictionary_hit'].value_counts())
headlines.loc[headlines['dictionary_hit'], ['source', 'source_type', 'headline', 'matched_terms']].sample(10, random_state=1)

## 1b. Dictionary hits by source

A dictionary is also a sampling diagnostic. If hits concentrate in one source or format, the dictionary may be measuring publication style as much as the target concept.

In [ ]:
headline_summary = (
    headlines
    .groupby(['source', 'source_type'])
    .agg(hit_rate=('dictionary_hit', 'mean'), n=('dictionary_hit', 'size'))
    .reset_index()
)

plt.figure(figsize=(9, 4))
sns.barplot(data=headline_summary, x='source', y='hit_rate', hue='source_type')
plt.title('Immigration-dictionary hit rate by source')
plt.ylabel('Share of headlines with dictionary hit')
plt.xlabel('')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()

headline_summary.sort_values('hit_rate', ascending=False)

Discussion: are the positives really about the concept? What obvious false negatives might the dictionary miss?

## 2. Labeled data: SMS spam

The SMS Spam Collection has human labels for short messages. It is a useful classroom dataset because it is local, imbalanced, and close to a real moderation/filtering task.

In [ ]:
sms = pd.read_csv(DATA_DIR / 'sms_spam.csv')
sms = sms[['text', 'label', 'is_spam']].dropna().copy()
sms['label'] = sms['label'].str.lower()

plt.figure(figsize=(5, 3.5))
sns.countplot(data=sms, x='label', order=['ham', 'spam'], color='#4C72B0')
plt.title('SMS label balance')
plt.xlabel('Human label')
plt.ylabel('Messages')
plt.tight_layout()

sms[['text', 'label']].head(), sms['label'].value_counts()

In [ ]:
spam_words = {
    'free', 'win', 'winner', 'won', 'prize', 'cash', 'urgent', 'claim',
    'txt', 'reply', 'call', 'mobile', 'ringtone', 'stop', 'guaranteed'
}


def dictionary_spam(text, threshold=1):
    tokens = spacy_preprocess(text, remove_stop=False, keep_alpha=True, min_len=1)
    score = sum(token in spam_words for token in tokens)
    return 'spam' if score >= threshold else 'ham'

sms['dict_prediction'] = sms['text'].apply(dictionary_spam)
display(sms[['text', 'label', 'dict_prediction']].head())
pd.crosstab(sms['label'], sms['dict_prediction'], normalize='index').round(3)

## 3. A small evaluation helper

This is intentionally short. The goal is to make model comparison readable, not to hide the work in a package.

### Methodology formulas: labels, prediction, and evaluation

A dictionary classifier maps text to a score by counting words from a theory-driven set $D$:

$$s_i = \sum_{w \in x_i} \mathbb{1}(w \in D), \qquad \hat{y}_i = \mathbb{1}(s_i > \tau).$$

Multinomial Naive Bayes chooses the class with the largest posterior log score:

$$\hat{c} = \arg\max_c \left[\log P(c) + \sum_v x_{iv}\log P(v \mid c)\right].$$

Logistic regression estimates a conditional probability from text features:

$$P(y_i=1 \mid x_i) = \sigma(\beta_0 + x_i^\top\beta), \qquad \sigma(z)=\frac{1}{1+e^{-z}}.$$

The notebook evaluates predictions with precision, recall, and F1:

$$\mathrm{precision}=\frac{TP}{TP+FP}, \qquad \mathrm{recall}=\frac{TP}{TP+FN}, \qquad F_1=2\frac{\mathrm{precision}\times\mathrm{recall}}{\mathrm{precision}+\mathrm{recall}}.$$

For an advanced class, the key point is that model accuracy is not the same as measurement validity: the learned label $\hat{y}$ is only useful if the training labels $y$ actually capture the target concept.


In [ ]:
def evaluate_predictions(y_true, y_pred, positive_label='spam'):
    return pd.Series({
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        'recall': recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        'f1': f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0)
    })

evaluate_predictions(sms['label'], sms['dict_prediction'])

In [ ]:
ConfusionMatrixDisplay.from_predictions(sms['label'], sms['dict_prediction'], cmap='Blues')
plt.title('Dictionary spam baseline')
plt.tight_layout()

## 4. Train and test split

Keep a held-out test set. We only look at test performance after training.

In [ ]:
train_df, test_df = train_test_split(
    sms,
    test_size=0.25,
    random_state=42,
    stratify=sms['label']
)

train_df.shape, test_df.shape

In [ ]:
nb_model = Pipeline([
    ('vectorizer', CountVectorizer(analyzer=spacy_analyzer, min_df=2, max_features=5000)),
    ('classifier', MultinomialNB())
])

logit_model = Pipeline([
    ('vectorizer', TfidfVectorizer(analyzer=spacy_analyzer_with_bigrams, min_df=2, max_features=7000)),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

nb_model.fit(train_df['text'], train_df['label'])
logit_model.fit(train_df['text'], train_df['label'])

test_df = test_df.copy()
test_df['nb_prediction'] = nb_model.predict(test_df['text'])
test_df['logit_prediction'] = logit_model.predict(test_df['text'])
test_df['dict_prediction'] = test_df['text'].apply(dictionary_spam)

In [ ]:
results = pd.DataFrame({
    'dictionary': evaluate_predictions(test_df['label'], test_df['dict_prediction']),
    'naive_bayes_counts': evaluate_predictions(test_df['label'], test_df['nb_prediction']),
    'logistic_tfidf': evaluate_predictions(test_df['label'], test_df['logit_prediction'])
}).T

results.round(3)

## 4a. Model comparison visuals

Tables are precise, but plots make tradeoffs visible. Here we compare metrics and precision-recall behavior across models.

In [ ]:
plt.figure(figsize=(8, 3.5))
sns.heatmap(results, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1)
plt.title('Held-out classification metrics')
plt.tight_layout()

In [ ]:
y_true_binary = (test_df['label'] == 'spam').astype(int)
model_scores = {
    'Naive Bayes': nb_model.predict_proba(test_df['text'])[:, list(nb_model.classes_).index('spam')],
    'Logistic TF-IDF': logit_model.predict_proba(test_df['text'])[:, list(logit_model.classes_).index('spam')]
}

plt.figure(figsize=(6, 5))
for name, scores in model_scores.items():
    precision, recall, _ = precision_recall_curve(y_true_binary, scores)
    ap = average_precision_score(y_true_binary, scores)
    plt.plot(recall, precision, label=f'{name} (AP={ap:.3f})')

baseline = y_true_binary.mean()
plt.axhline(baseline, color='gray', linestyle='--', label=f'Baseline spam rate={baseline:.2f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-recall curves for spam detection')
plt.legend()
plt.tight_layout()

### Additional demo: calibration and threshold choice

The same predicted probabilities can support different decisions. Here the threshold is the probability at which a message is sent to the spam folder, so false positives and false negatives have different substantive costs.

In [ ]:
from sklearn.calibration import calibration_curve

logit_scores = model_scores['Logistic TF-IDF']
prob_true, prob_pred = calibration_curve(y_true_binary, logit_scores, n_bins=8, strategy='quantile')

threshold_rows = []
for threshold in np.linspace(0.20, 0.80, 7):
    pred = (logit_scores >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'precision': precision_score(y_true_binary, pred, zero_division=0),
        'recall': recall_score(y_true_binary, pred, zero_division=0),
        'f1': f1_score(y_true_binary, pred, zero_division=0),
        'positive_rate': pred.mean()
    })
threshold_table = pd.DataFrame(threshold_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
axes[0].plot(prob_pred, prob_true, marker='o', label='Logistic TF-IDF')
axes[0].set_title('Calibration curve')
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Observed positive rate')
axes[0].legend()

threshold_long = threshold_table.melt(
    id_vars='threshold',
    value_vars=['precision', 'recall', 'f1', 'positive_rate'],
    var_name='metric',
    value_name='value'
)
sns.lineplot(data=threshold_long, x='threshold', y='value', hue='metric', marker='o', ax=axes[1])
axes[1].set_title('Changing the classification threshold')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Metric value')

plt.tight_layout()
threshold_table.round(3)

In [ ]:
ConfusionMatrixDisplay.from_predictions(test_df['label'], test_df['logit_prediction'], cmap='Blues')
plt.title('Logistic regression with TF-IDF')
plt.tight_layout()

## 5. Error analysis

Read errors before trusting a model. Errors are often substantively informative.

In [ ]:
def show_errors(
    df,
    prediction_col,
    true_col='label',
    text_col='text',
    n=8,
    kind='false_positive',
    positive_label='spam',
    negative_label='ham'
):
    if kind == 'false_positive':
        subset = df[(df[true_col] == negative_label) & (df[prediction_col] == positive_label)]
    elif kind == 'false_negative':
        subset = df[(df[true_col] == positive_label) & (df[prediction_col] == negative_label)]
    else:
        subset = df[df[true_col] != df[prediction_col]]
    return subset[[text_col, true_col, prediction_col]].sample(min(n, len(subset)), random_state=2)

show_errors(test_df, 'logit_prediction', kind='false_positive')

In [ ]:
show_errors(test_df, 'logit_prediction', kind='false_negative')

## 6. What did the classifier learn?

For this binary logistic classifier, positive coefficients point toward spam and negative coefficients point toward ham.

In [ ]:
feature_names = logit_model.named_steps['vectorizer'].get_feature_names_out()
coefs = logit_model.named_steps['classifier'].coef_[0]

features = pd.DataFrame({'term': feature_names, 'coefficient': coefs})
pd.concat([
    features.sort_values('coefficient', ascending=False).head(15),
    features.sort_values('coefficient', ascending=True).head(15)
])

## 6a. Coefficient plot

Feature weights are not causal effects, but they help diagnose what a classifier is using to separate classes.

In [ ]:
coef_plot = pd.concat([
    features.sort_values('coefficient', ascending=True).head(15),
    features.sort_values('coefficient', ascending=False).head(15)
]).sort_values('coefficient')

plt.figure(figsize=(8, 8))
colors = np.where(coef_plot['coefficient'] > 0, '#4C72B0', '#C44E52')
plt.barh(coef_plot['term'], coef_plot['coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=1)
plt.title('Most influential TF-IDF terms in spam classifier')
plt.xlabel('Coefficient toward spam')
plt.tight_layout()

### Additional demo: annotation budget learning curve

A learning curve connects model performance to the amount of labeled data. This is a practical way to discuss whether more annotation is likely to help.

In [ ]:
learning_seed = 42
candidate_sizes = [50, 100, 250, 500, 1000, min(2000, len(train_df))]
train_sizes = sorted({size for size in candidate_sizes if size <= len(train_df)})


def balanced_sample(frame, label_col, size, seed):
    per_class = max(1, size // frame[label_col].nunique())
    pieces = []
    for _, group in frame.groupby(label_col):
        pieces.append(group.sample(min(per_class, len(group)), random_state=seed))
    return pd.concat(pieces).sample(frac=1, random_state=seed)

learning_rows = []
for size in train_sizes:
    subset = balanced_sample(train_df, 'label', size, learning_seed + size)
    model = Pipeline([
        ('vectorizer', TfidfVectorizer(analyzer=spacy_analyzer_with_bigrams, min_df=2, max_features=3000)),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])
    model.fit(subset['text'], subset['label'])
    pred = model.predict(test_df['text'])
    learning_rows.append({
        'labeled_examples': len(subset),
        'accuracy': accuracy_score(test_df['label'], pred),
        'f1': f1_score(test_df['label'], pred, pos_label='spam', zero_division=0)
    })

learning_curve = pd.DataFrame(learning_rows)

plt.figure(figsize=(6, 4))
learning_long = learning_curve.melt(id_vars='labeled_examples', var_name='metric', value_name='value')
sns.lineplot(data=learning_long, x='labeled_examples', y='value', hue='metric', marker='o')
plt.ylim(0, 1)
plt.title('Learning curve for SMS spam data')
plt.xlabel('Number of labeled training examples')
plt.ylabel('Held-out performance')
plt.tight_layout()

learning_curve.round(3)

## 7. Optional extension: Federalist authorship attribution

Train on essays known to be by Hamilton or Madison, then predict disputed essays. This connects authorship attribution to the same classification workflow.

In [ ]:
federalist = pd.read_csv(DATA_DIR / 'federalist.csv')

known = federalist[federalist['author'].isin(['HAMILTON', 'MADISON'])].copy()
disputed = federalist[federalist['author'].eq('HAMILTON OR MADISON')].copy()

author_model = Pipeline([
    ('vectorizer', CountVectorizer(analyzer=spacy_analyzer, min_df=2, max_features=3000)),
    ('classifier', MultinomialNB())
])

author_model.fit(known['text'], known['author'])
disputed['predicted_author'] = author_model.predict(disputed['text'])

disputed[['paper_id', 'title', 'author', 'predicted_author']]

## 8. Student task

Choose one concept and one model. Then write a short validation note:

1. What is the concept?
2. What is the label or proxy?
3. What errors matter most?
4. What would you inspect before using the model in a paper?

In [ ]:
# Your turn: edit the dictionary words or vectorizer settings, then rerun evaluation.
custom_spam_words = spam_words | {'voucher', 'bonus', 'selected'}

# Keep this cell as a workspace for your own model comparison.